Cell 1: Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import time
sys.path.append("..")

from src.generation import fact_check

/Users/simgecevik/Desktop/WT/teyit-asistani/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1027 17:22:51.525000 23760 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Cell 2:Five test cases

In [2]:
cases = [
    # (claim, article to exclude, expected verdict)
    ("Amazon teslim edilemeyen paketleri 2 euroya satıyormuş", None, "FALSE"),
    ("Amazon Türkiye'nin sahipsiz paketleri ucuza sattığı bir uygulaması yok", None, "TRUE"),
    ("Threads hesabını tamamen silmek için Instagram hesabını da silmek gerekiyor", None, "TRUE"),
    ("Amazon teslim edilemeyen paketleri 2 euroya satıyormuş", "fctr-2", "NOT_ENOUGH_EVIDENCE"),
    ("Ankara'da ekim ayında kar yağdı", None, "NOT_ENOUGH_EVIDENCE"),
]

for claim, exclude, expected in cases:
    result, evidence = fact_check(claim, exclude_doc_id=exclude)
    mark = "+" if result.verdict == expected else "-"

    print(f"{mark} {claim}" + (f" (excluded {exclude})" if exclude else ""))
    print(f"    expected: {expected} | got: {result.verdict}")
    for a in result.evidence_assessments:
        if 1 <= a.evidence_id <= len(evidence):
            item = evidence[a.evidence_id - 1]
            print(f"   evidence {a.evidence_id}: same claim = {a.is_same_claim} | "
                  f"{item['label']:8} | {item['claim'][:60]}")
    print(f"    explanation: {result.explanation}")
    print("-" * 100)
    time.sleep(5) # Flash-Lite allows 15 requests per minute


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6371.45it/s]


+ Amazon teslim edilemeyen paketleri 2 euroya satıyormuş
    expected: FALSE | got: FALSE
   evidence 1: same claim = True | yanlış   | Amazon'un Türkiye'de teslim edilmemiş ürünleri uygun fiyata 
   evidence 2: same claim = False | doğru    | Amazon'da Türk bayraklı klozet takımlarının satıldığı iddias
   evidence 3: same claim = False | doğru    | Almanya’da Lidl’ın 599 euroya Türkiye’de tatil sattığı iddia
    explanation: Teyit'in 11 Temmuz 2023 tarihli analizine göre, Amazon'un sahiplerine ulaştırılamayan paketleri 2 euroya sattığı iddiası gerçeği yansıtmamakta olup bir dolandırıcılık yöntemidir. Bu durumun o tarihten sonra değişmiş olabileceği göz önünde bulundurulmalıdır.
----------------------------------------------------------------------------------------------------
+ Amazon Türkiye'nin sahipsiz paketleri ucuza sattığı bir uygulaması yok
    expected: TRUE | got: TRUE
   evidence 1: same claim = True | yanlış   | Amazon'un Türkiye'de teslim edilmemiş ürünleri uygun fiyata 


## Findings

- All 5 hand-picked cases return the expected verdict, including two "related but
  not the same claim" traps (Amazon toilet seat covers, Ankara snow photos).
- Per-evidence `is_same_claim` judgements make each verdict explainable.
- Explanations cite the source and date and warn that the situation may have changed.
- Date format in explanations is inconsistent (ISO vs Turkish); to be fixed in the UI.
- Flash-Lite occasionally makes small wording errors ("idaresini" instead of "iddiasını").
- Not yet known whether NOT_ENOUGH_EVIDENCE came from the model or the guardrail
  → to be tracked in evaluation.
- These are hand-picked cases, not a systematic evaluation.